# 量化对照：双均线 10/20 vs MACD vs 一直拿着

一次跑三个对照，都用 backtrader，同一段行情、同一笔手续费（0.1%），看谁真的强。

只用公开行情，不真交易。实盘接口在最后，只留口子，不填真 Key，什么时候接真钱由你决定。

注意：pandas 简化版算出来的 MACD 很高是假的，那是没有预热期算进去的；这里以 backtrader 为准。

In [ ]:
!pip install -q ccxt backtrader pandas matplotlib

In [ ]:
import ccxt, pandas as pd, requests

# 拿行情：先用 ccxt（币安美国站公开接口，不用 Key），不通就改直连
try:
    ex = ccxt.binanceus()
    bars = ex.fetch_ohlcv("BTC/USDT", timeframe="1d", limit=1000)
    df = pd.DataFrame(bars, columns=["time", "open", "high", "low", "close", "volume"])
    print("用 ccxt 拿到行情")
except Exception as e:
    print("ccxt 不通，改用直连：", e)
    data = requests.get("https://api.binance.us/api/v3/klines",
        params={"symbol": "BTCUSDT", "interval": "1d", "limit": 1000}, timeout=30).json()
    df = pd.DataFrame(data, columns=["time", "open", "high", "low", "close", "volume",
        "close_time", "quote_vol", "trades", "taker_buy_base", "taker_buy_quote", "ignore"])
    df = df[["time", "open", "high", "low", "close", "volume"]]

df["date"] = pd.to_datetime(df["time"], unit="ms")
df[["open", "high", "low", "close", "volume"]] = df[["open", "high", "low", "close", "volume"]].astype(float)
df = df.set_index("date")
print("行情天数:", len(df))
print("最新收盘:", df["close"].iloc[-1])
print(f"一直拿着：{df['close'].iloc[-1] / df['close'].iloc[0] - 1:.1%}")

In [ ]:
import backtrader as bt

class DualMA(bt.Strategy):
    params = dict(fast=10, slow=20)

    def __init__(self):
        self.ma_fast = bt.ind.SMA(self.data.close, period=self.p.fast)
        self.ma_slow = bt.ind.SMA(self.data.close, period=self.p.slow)
        self.crossover = bt.ind.CrossOver(self.ma_fast, self.ma_slow)

    def next(self):
        if not self.position:
            if self.crossover > 0:
                self.buy()
        else:
            if self.crossover < 0:
                self.sell()

class MACDStrat(bt.Strategy):
    params = dict(me1=19, me2=39, sig=9)

    def __init__(self):
        self.macd = bt.ind.MACD(self.data.close, period_me1=self.p.me1,
                                period_me2=self.p.me2, period_signal=self.p.sig)
        self.crossover = bt.ind.CrossOver(self.macd.macd, self.macd.signal)

    def next(self):
        if not self.position:
            if self.crossover > 0:
                self.buy()
        else:
            if self.crossover < 0:
                self.sell()

def run(strat, name):
    cerebro = bt.Cerebro()
    cerebro.addstrategy(strat)
    cerebro.adddata(bt.feeds.PandasData(dataname=df))
    cerebro.broker.setcash(10000.0)
    cerebro.broker.setcommission(commission=0.001)
    cerebro.addsizer(bt.sizers.PercentSizer, percents=95)
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name="dd")
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name="trades")
    start = cerebro.broker.getvalue()
    results = cerebro.run()
    end = cerebro.broker.getvalue()
    strat = results[0]
    print(f"{name}：{start:.0f} -> {end:.0f}，收益率 {(end / start - 1):.1%}，" 
          f"最大回撤 {strat.analyzers.dd.get_analysis()['max']['drawdown']:.1f}%，" 
          f"交易 {strat.analyzers.trades.get_analysis()['total']['total']} 笔，" 
          f"现在 {'持有' if strat.position else '空仓'}")

run(DualMA, "双均线 10/20")
run(MACDStrat, "MACD 19/39/9")

## 实盘接口（预留，没开启）

下面这段是留给以后对接交易所的口子，现在是关着的，不会下单、不会动钱。

真要接的时候再说：只用测试网或者只读 Key，先小额试，Key 绝不写进公开的本子里。

In [ ]:
# 实盘开关，现在是关的
LIVE_TRADING = False
API_KEY = ""   # 以后填，不写进公开仓库
API_SECRET = ""

if LIVE_TRADING:
    print("实盘已开启，会下单")
else:
    print("实盘未开启，只做回测，不动钱")

这只是演示，回测不代表未来，不构成投资建议，也没有真交易。

2026-11-08 本地验算：同一段行情（2024-02-13 到 2026-11-08），一直拿着 +52.4%，双均线 10/20 是 +12.3%，MACD 19/39/9 是 +47.8%，两个都是空仓。MACD 回撤小一点，但还是没跑赢一直拿着。